In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import cv2
import torchvision.transforms as transforms
from PIL import Image

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'E:\DATA\Annotations',
    'video_root': r'E:\DATA\Videos', 
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'target_label': 'bangla-tesla', 
    'obs_len': 15,  
    'pred_len': 45, 
    
    # MTN Specifics
    'img_size': (64, 64), # Resize vehicle crops to 64x64
    'hidden_size': 256,
    'embed_size': 64,
    'batch_size': 16,     # Smaller batch size because images take memory
    'epochs': 20,
    'lr': 5e-4
}

# Image Preprocessing (Standard ImageNet Normalization)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"✅ MTN Config Loaded. Device: {CONFIG['device']}")

✅ MTN Config Loaded. Device: cpu


In [3]:
class IDDMultiModalDataset(Dataset):
    def __init__(self, xml_root, video_root, obs_len=15, pred_len=45, target_label='bangla-tesla', transform=None):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        self.video_root = video_root
        self.transform = transform
        
        print(f"📂 Parsing XMLs for label: '{target_label}'...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                meta_size = root.find('meta').find('original_size')
                img_w = float(meta_size.find('width').text)
                img_h = float(meta_size.find('height').text)
                
                # Infer Video Path: "video1.xml" -> "video1.mp4"
                filename = os.path.basename(xml).replace('.xml', '.mp4')
                video_path = os.path.join(self.video_root, filename)
                
                for track in root.findall('track'):
                    if track.attrib['label'] != target_label: continue
                    
                    raw_boxes = [] # Store [frame, xtl, ytl, xbr, ybr] for image extraction
                    norm_data = [] # Store [cx, cy, w, h] for LSTM
                    
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    
                    for box in boxes:
                        if box.get('outside') == '1': continue
                        frame_idx = int(box.attrib['frame'])
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        
                        w = xbr - xtl
                        h = ybr - ytl
                        cx = (xtl + xbr) / 2
                        cy = (ytl + ybr) / 2
                        
                        raw_boxes.append([frame_idx, xtl, ytl, xbr, ybr])
                        norm_data.append([cx/img_w, cy/img_h, w/img_w, h/img_h])
                    
                    norm_data = np.array(norm_data)
                    raw_boxes = np.array(raw_boxes)
                    
                    if len(norm_data) < self.seq_len: continue
                    
                    stride = 10 
                    for i in range(0, len(norm_data) - self.seq_len + 1, stride):
                        obs = norm_data[i : i+obs_len]
                        pred = norm_data[i+obs_len : i+obs_len+pred_len]
                        
                        # Get box info for the LAST observed frame (t=15)
                        last_obs_idx = i + obs_len - 1
                        target_box = raw_boxes[last_obs_idx] 
                        
                        self.samples.append({
                            'obs': obs[:, 0:2],
                            'pred': pred,
                            'video_path': video_path,
                            'box_info': target_box
                        })
            except Exception as e: pass

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # 1. Coordinate Data
        obs_tensor = torch.tensor(item['obs'], dtype=torch.float32)
        pred_tensor = torch.tensor(item['pred'], dtype=torch.float32)
        
        # 2. Image Data
        video_path = item['video_path']
        frame_id, xtl, ytl, xbr, ybr = item['box_info']
        
        # Default black image if video fails
        img_tensor = torch.zeros(3, CONFIG['img_size'][0], CONFIG['img_size'][1])
        
        if os.path.exists(video_path):
            cap = cv2.VideoCapture(video_path)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
            ret, frame = cap.read()
            cap.release()
            
            if ret:
                h_img, w_img, _ = frame.shape
                # Crop safe bounds
                y1, y2 = max(0, int(ytl)), min(h_img, int(ybr))
                x1, x2 = max(0, int(xtl)), min(w_img, int(xbr))
                
                if y2 > y1 and x2 > x1:
                    crop = frame[y1:y2, x1:x2]
                    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                    if self.transform:
                        img_tensor = self.transform(crop)

        return obs_tensor, img_tensor, pred_tensor

# Create Dataset
dataset = IDDMultiModalDataset(
    CONFIG['xml_root'], 
    CONFIG['video_root'],
    obs_len=CONFIG['obs_len'], 
    pred_len=CONFIG['pred_len'],
    target_label=CONFIG['target_label'],
    transform=transform
)
print(f"✅ MTN Dataset Created: {len(dataset)} sequences.")

📂 Parsing XMLs for label: 'bangla-tesla'...


  0%|          | 0/6 [00:00<?, ?it/s]

✅ MTN Dataset Created: 2156 sequences.


In [4]:
class VisualEncoder(nn.Module):
    def __init__(self, embed_size=64):
        super(VisualEncoder, self).__init__()
        # Simple 3-Layer CNN
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten()
        )
        self.fc = nn.Linear(128, embed_size)
        
    def forward(self, x):
        return self.fc(self.cnn(x))

class MTN(nn.Module):
    def __init__(self, input_size=2, output_size=4, hidden_size=256, embed_size=64):
        super(MTN, self).__init__()
        
        # Modules
        self.visual_net = VisualEncoder(embed_size)
        self.coord_embed = nn.Linear(input_size, embed_size)
        self.encoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        
        # Fusion: Combines Visual (64) + LSTM Hidden (256)
        self.fusion_layer = nn.Linear(hidden_size + embed_size, hidden_size)
        
        # Decoder
        self.decoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.pred_fc = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, obs, images, pred_len):
        # 1. Encode Coordinates
        obs_emb = self.relu(self.coord_embed(obs))
        _, (h_enc, c_enc) = self.encoder_lstm(obs_emb)
        
        # 2. Encode Images
        vis_emb = self.visual_net(images) # [Batch, 64]
        
        # 3. Fuse
        h_enc_sq = h_enc.squeeze(0) # [Batch, 256]
        fused = torch.cat((h_enc_sq, vis_emb), dim=1) # [Batch, 320]
        
        # Init Decoder State
        h_dec = self.relu(self.fusion_layer(fused)).unsqueeze(0)
        c_dec = c_enc # Pass cell state through
        
        # 4. Decode
        outputs = []
        curr_input = obs[:, -1, :].unsqueeze(1)
        
        h, c = h_dec, c_dec
        
        for _ in range(pred_len):
            curr_emb = self.relu(self.coord_embed(curr_input))
            out, (h, c) = self.decoder_lstm(curr_emb, (h, c))
            pred_step = self.pred_fc(out)
            outputs.append(pred_step)
            curr_input = pred_step[:, :, :2]
            
        return torch.cat(outputs, dim=1)

# Initialize
model = MTN(hidden_size=CONFIG['hidden_size'], embed_size=CONFIG['embed_size']).to(CONFIG['device'])
print(f"✅ MTN Model Initialized.")

✅ MTN Model Initialized.


In [ ]:
# Split Data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

# num_workers=0 is safer for OpenCV on Windows
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
criterion = nn.MSELoss()

print(f"🚀 Starting MTN Training...")

for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss = 0
    
    # Note: Loop now returns Tuple of 3 (obs, img, target)
    for obs, img, target in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}"):
        obs = obs.to(CONFIG['device'])
        img = img.to(CONFIG['device'])
        target = target.to(CONFIG['device'])
        
        optimizer.zero_grad()
        
        preds = model(obs, img, CONFIG['pred_len'])
        loss = criterion(preds, target)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for obs, img, target in val_loader:
            obs = obs.to(CONFIG['device'])
            img = img.to(CONFIG['device'])
            target = target.to(CONFIG['device'])
            
            preds = model(obs, img, CONFIG['pred_len'])
            val_loss += criterion(preds, target).item()
            
    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.5f} | Val Loss: {val_loss/len(val_loader):.5f}")

torch.save(model.state_dict(), "bangla_tesla_mtn.pth")
print("💾 MTN Model Saved.")